In [1]:
import pandas as pd
from pathlib import Path

# Definir rutas base
BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_VENTAS = BASE / "data" / "fuente" / "ventas"
DIR_MAESTROS_V1 = BASE / "data" / "fuente" / "maestros" / "v1"
DIR_MAESTROS_V2 = BASE / "data" / "fuente" / "maestros" / "v2"

In [2]:
# 2. Leer y consolidar todos los archivos CSV de ventas diarios en un solo DataFrame
archivos_ventas = list(DIR_VENTAS.glob("ventas_*.csv"))
df_ventas = pd.concat((pd.read_csv(f, dtype=str, keep_default_na=False) for f in archivos_ventas), ignore_index=True)

print(f"Filas antes de limpieza: {len(df_ventas)}")


Filas antes de limpieza: 504749


In [3]:
# 3. Limpieza de Consistencia
# a) SKUs en minúsculas y con espacios
df_ventas["sku"] = df_ventas["sku"].str.strip().str.upper()

# b) Canal con otra grafía
mapeo_canal = {"Tienda fisica": "Tienda física", "ecommerce": "E-commerce", "APP": "App"}
df_ventas["canal"] = df_ventas["canal"].replace(mapeo_canal)

# c) Sucursal vacía
df_ventas.loc[df_ventas["sucursal"] == "", "sucursal"] = "Desconocida"

# d) RUT sin puntos (ej. 12345678-5 -> 12.345.678-5)
def formatear_rut(rut):
    if not rut or "-" not in rut: return rut
    cuerpo, dv = rut.split("-")
    if "." not in cuerpo: # Si no tiene puntos, se los agregamos
        cuerpo = f"{int(cuerpo):,}".replace(",", ".")
    return f"{cuerpo}-{dv}"

df_ventas["rut_cliente"] = df_ventas["rut_cliente"].apply(formatear_rut)

# e) SKU y Campaña inexistente
# Cargamos los maestros reales para validar cuáles existen realmente
productos_validos = pd.read_json(DIR_MAESTROS_V2 / "productos.json")["sku"].unique()
campanas_validas = pd.read_csv(DIR_MAESTROS_V2 / "campanas.csv")["cod_campana"].unique()

df_ventas.loc[~df_ventas["sku"].isin(productos_validos), "sku"] = "DESC-000" # SKU por defecto
df_ventas.loc[~df_ventas["cod_campana"].isin(campanas_validas) & (df_ventas["cod_campana"] != ""), "cod_campana"] = ""


In [4]:
# 4. Limpieza de Validez
# a) Fechas con '/' e imposibles
df_ventas["fecha"] = df_ventas["fecha"].str.replace("/", "-")
df_ventas["fecha"] = pd.to_datetime(df_ventas["fecha"], errors="coerce") # Convierte fechas malas (ej. 2025-02-30) a NaT
df_ventas = df_ventas.dropna(subset=["fecha"]) # Eliminamos las filas con fechas imposibles

# b) Montos no numéricos y negativos
df_ventas["monto"] = pd.to_numeric(df_ventas["monto"], errors="coerce")
df_ventas = df_ventas.dropna(subset=["monto"]) # Eliminamos los montos "N/A"
df_ventas["monto"] = df_ventas["monto"].abs() # Convertimos negativos a positivos absolutos

In [5]:
# 5. Limpieza de Unicidad
# Eliminar líneas duplicadas conservando la primera aparición
df_ventas = df_ventas.drop_duplicates(subset=["id_venta"], keep="first")

print(f"Filas después de limpieza: {len(df_ventas)}")

Filas después de limpieza: 498753
